# Limpieza de los datos

- **Limpieza:** pasar a nulo los valores inválidos y corregir las inconsistencias de formato.
- **Filtrado:** sacar las escuchas que son imposibles
- **Selección de atributos:** quedarnos solo con las columnas que se usan

Todavía no hacemos **agregaciones ni atributos derivados** (promedios, rankings, similitudes).
Eso depende de cada pregunta y va en un notebook aparte.

No se elimina ningún usuario ni ninguna canción. Los problemas se tratan con nulos, para no perder
escuchas que sirven para otros análisis.

## Contenido
1. Preparación del entorno
2. Qué vamos a limpiar
3. Usuarios
4. Audio features
5. Hofstede
6. Happiness
7. Escuchas
8. Verificación general

## 1. Preparación del entorno

In [2]:
import duckdb                    # motor SQL que lee archivos parquet sin cargarlos enteros en memoria
import pandas as pd              # para trabajar con tablas chicas
from pathlib import Path         # para crear carpetas

# Carpetas
# Buscamos la carpeta data subiendo desde la carpeta donde está el notebook.
# Así funciona esté en la raíz del repo o en una subcarpeta (visualización, minería, etc.).
carpeta = Path.cwd()
while not (carpeta / "data" / "parquet").exists():
    if carpeta == carpeta.parent:
        raise FileNotFoundError("No encontramos la carpeta data/parquet. Revisar dónde está.")
    carpeta = carpeta.parent

ruta = (carpeta / "data" / "parquet").as_posix() + "/"        # datos originales
ruta_clean = (carpeta / "data" / "clean").as_posix() + "/"    # acá guardamos los datos limpios
tmp = carpeta / ".tmp"                                        # carpeta temporal de DuckDB
Path(ruta_clean).mkdir(parents=True, exist_ok=True)
tmp.mkdir(exist_ok=True)

# Conexion a DuckDB
con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")                # tope de memoria
con.execute(f"SET temp_directory = '{tmp.as_posix()}'")  # si una consulta necesita más memoria, usa esta carpeta del disco
con.execute("SET preserve_insertion_order = false")    # ahorra memoria; no necesitamos mantener el orden de las filas
con.execute("SET TimeZone = 'UTC'")                    # se usa la misma zona horaria

# Vistas, usamos un nombre corto para cada archivo
for tabla in ["events", "users", "artists", "acoustic_features", "hofstede", "happiness"]:
    con.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM '{ruta}{tabla}.parquet'")

# funcion auxiliar
def q(consulta):
    """
    ejecuta una consulta SQL y devuelve el resultado como DataFrame de pandas.
    Parametro: consulta (str): consulta SQL a ejecutar.
    Retorna: pd.DataFrame: resultado de la consulta.
    """
    return con.sql(consulta).df()

def guardar(consulta, nombre):
    """
    Ejecuta una consulta SQL y guarda el resultado como archivo Parquet en la carpeta data/clean/.
    Parámetros:
        consulta (str): consulta SQL a ejecutar.
        nombre (str): nombre del archivo sin la extensión.
    Retorna: None.
    """
    # COPY escribe directo a disco, así que no hace falta que el resultado entre en memoria.
    destino = f"{ruta_clean}{nombre}.parquet"
    con.execute(f"COPY ({consulta}) TO '{destino}' (FORMAT PARQUET)")
    filas = con.execute(f"SELECT COUNT(*) FROM '{destino}'").fetchone()[0]
    print(f"{nombre}: {filas:,} filas guardadas en {destino}")

def nulos_por_columna(nombre):
    """
    Cuenta la cantidad de valores nulos de cada columna de un archivo limpio.
    Parámetro: nombre (str): nombre del archivo sin la extensión.
    Retorna: pd.DataFrame: cantidad de nulos por columna.
    """
    archivo = f"{ruta_clean}{nombre}.parquet"
    columnas = q(f"DESCRIBE SELECT * FROM '{archivo}'")["column_name"]
    partes = [f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in columnas]
    resultado = q(f"SELECT {', '.join(partes)} FROM '{archivo}'").T
    resultado.columns = ["nulos"]
    return resultado


| Archivo | Problema | Operación | Qué hacemos |
|---|---|---|---|
| `users_clean` | Edades fuera de 10 a 90 años (incluido -1) | Limpieza | Las pasamos a nulo |
| | Género "n" o vacío | Limpieza | Dejamos ambos como "no informado" |
| | País con código "UK" | Limpieza | Cambiamos "UK" por "GB" |
| | `playcount` y `registered_unixtime` | Selección de atributos | Sacamos las dos columnas |
| `acoustic_features_clean` | Tempo en 0 | Limpieza | Lo pasamos a nulo |
| | Loudness en -60 dB | Limpieza | Lo pasamos a nulo |
| `hofstede_clean` | Dimensiones en formato texto | Limpieza | Las pasamos a numérico (int)|
| | País con código "UK" | Limpieza | Cambiamos "UK" por "GB" |
| | Diferencias en los nombres de los países | Limpieza | Unificamos los nombres |
| `happiness_clean` | Falta el código de país | Limpieza | Agregamos el código de dos letras |
| | Años que no coinciden con los de las escuchas | Filtrado | Nos quedamos con 2005–2014 |
| | Indicadores que no usamos | Selección de atributos | Dejamos `Life Ladder` como único indicador |
| `events_clean` | Una escucha en 1970 | Filtrado | Eliminamos esa escucha |
| | Más de 720 escuchas por usuario en un día | Filtrado | Eliminamos las escuchas de esos días |
| | `album_id` | Selección de atributos | Sacamos la columna |


Los nulos en las características de audio y los usuarios sin escuchas se mantienen como estan

## 3. Usuarios

Sacamos la columna `playcount` porque en el README aclaran que no coincide 
con los datos de las escuchas. También sacamos `registered_unixtime` porque
no la necesitamos para armar los gráficos.

In [3]:
#Usamos CASE WHEN para limpiar los datos según distintas condiciones. 
#En el caso de la edad, si está fuera del rango de 10 a 90 años queda como nulo.
guardar("""
    SELECT user_id,
           CASE WHEN country = 'UK' THEN 'GB' ELSE country END            AS country,
           CASE WHEN age BETWEEN 10 AND 90 THEN age END                   AS age,
           CASE WHEN gender IN ('m', 'f') THEN gender ELSE 'no informado' END AS gender
    FROM users
""", "users_clean")

users_clean: 55,190 filas guardadas en c:/Users/mgrau/musica-por-pais/data/clean/users_clean.parquet


In [4]:
#Control de la limpieza, para comparar con los valores que obtuvimos en la exploración de datos

archivo = ruta_clean + "users_clean.parquet"

#Edad
display(q(f"SELECT COUNT(*) - COUNT(age) AS edades_nulas, COUNT(*) AS usuarios FROM '{archivo}'"))

#Género
display(q(f"SELECT gender, COUNT(*) AS usuarios FROM '{archivo}' GROUP BY gender ORDER BY usuarios DESC"))

# País no debería quedar ningún "UK" y "GB"
display(q(f"SELECT country, COUNT(*) AS usuarios FROM '{archivo}' WHERE country IN ('UK', 'GB') GROUP BY country"))

,edades_nulas,usuarios
0,12481,55190


,gender,usuarios
0,m,36506
1,f,13937
2,no informado,4747


,country,usuarios
0,GB,4534


## 4. Audio features

In [5]:
#Un tempo de 0 BPM y un volumen de -60 dB son valores que Spotify no pudo calcular, así que los
#pasamos a nulo
#  SELECT * REPLACE devuelve todas las columnas, pero cambia solo las que nombramos.
guardar("""
    SELECT * REPLACE ( 
        NULLIF(tempo, 0)                              AS tempo,
        CASE WHEN loudness = -60 THEN NULL ELSE loudness END AS loudness
    )
    FROM acoustic_features
""", "acoustic_features_clean")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

acoustic_features_clean: 3,471,884 filas guardadas en c:/Users/mgrau/musica-por-pais/data/clean/acoustic_features_clean.parquet


In [6]:
q(f"""
    SELECT COUNT(*) FILTER (WHERE tempo IS NULL)    AS tempo_nulo,
           COUNT(*) FILTER (WHERE tempo = 0)        AS tempo_cero,
           COUNT(*) FILTER (WHERE loudness IS NULL) AS loudness_nulo,
           COUNT(*) FILTER (WHERE loudness = -60)   AS loudness_silencio
    FROM '{ruta_clean}acoustic_features_clean.parquet'
""")

,tempo_nulo,tempo_cero,loudness_nulo,loudness_silencio
0,2651,0,209,0


## 5. Hofstede

Para unir las tablas por país usamos siempre la columna `country` con el código de 2 letras.
El nombre queda en `country_name`.


Eliminar para la entrega lo agrego solo para entender la query

CAST: convierte un dato a otro tipo. Si no puede convertirlo, devuelve un error.
TRY_CAST: intenta convertir el dato. Si no puede, devuelve un nulo.
NULLIF + CAST: reemplaza un valor específico por nulo antes de convertirlo.
CASE WHEN + CAST: permite definir condiciones para decidir qué valores convertir y cuáles dejar como nulos.

::: es una forma abreviada de escribir CAST.

In [7]:
#Las dimensiones están guardadas como texto, así que las convertimos a número con TRY_CAST.
#Si algún valor no se puede convertir,TRY_CAST devuelve nulo en vez de dar error.
#Corregimos los nombres mal escritos o abreviados
guardar("""
    SELECT CASE WHEN h.ctr = 'UK' THEN 'GB' ELSE h.ctr END AS country,
           CASE h.country
               WHEN 'Bulgeria'      THEN 'Bulgaria'
               WHEN 'U.S.A.'        THEN 'United States'
               WHEN 'Czech Rep'     THEN 'Czech Republic'
               WHEN 'Slovak Rep'    THEN 'Slovakia'
               WHEN 'Great Britain' THEN 'United Kingdom'
               ELSE h.country
           END AS country_name,
           TRY_CAST(h.power_distance AS DOUBLE)       AS power_distance,
           TRY_CAST(h.individualism AS DOUBLE)        AS individualism,
           TRY_CAST(h.masculinity AS DOUBLE)          AS masculinity,
           TRY_CAST(h.uncertainty_avoidance AS DOUBLE) AS uncertainty_avoidance,
           CAST(h.long_term_orientation AS DOUBLE)    AS long_term_orientation,
           TRY_CAST(h.indulgence AS DOUBLE)           AS indulgence
    FROM hofstede h
""", "hofstede_clean")

hofstede_clean: 47 filas guardadas en c:/Users/mgrau/musica-por-pais/data/clean/hofstede_clean.parquet


In [8]:
#Controlamos cuántos valores nulos quedaron después de convertir las
#dimensiones a números y revisamos los nombres y códigos de los países

display(nulos_por_columna("hofstede_clean"))
display(q(f"SELECT country, country_name, indulgence FROM '{ruta_clean}hofstede_clean.parquet' ORDER BY country_name LIMIT 10"))

,nulos
country,0
country_name,0
power_distance,2
individualism,2
masculinity,2
uncertainty_avoidance,2
long_term_orientation,0
indulgence,1


,country,country_name,indulgence
0,AR,Argentina,62.0
1,AU,Australia,71.0
2,AT,Austria,63.0
3,BY,Belarus,15.0
4,BE,Belgium,57.0
5,BR,Brazil,59.0
6,BG,Bulgaria,16.0
7,CA,Canada,68.0
8,CL,Chile,68.0
9,CN,China,24.0


## 6. Happiness

Como esta tabla solo tiene el nombre de los países, usamos la librería
pycountry para obtener el código de dos letras y poder relacionarla
con las otras tablas. Los nombres que la librería no reconoce los
corregimos manualmente.

Filtramos los años de 2005 a 2014 para trabajar con el mismo período
que las escuchas y nos quedamos con Life Ladder como indicador de felicidad.

Por ahora mantenemos los datos de cada país por año, sin calcular promedios.

In [ ]:
import pycountry   # si da error, instalar con: pip install pycountry

# Nombres que pycountry no reconoce o reconoce distinto
EXCEPCIONES = {
    "Russia": "RU",
    "South Korea": "KR",
    "Taiwan Province of China": "TW",
    "Hong Kong S.A.R. of China": "HK",
    "Iran": "IR",
    "Vietnam": "VN",
    "Venezuela": "VE",
    "Bolivia": "BO",
    "Moldova": "MD",
    "Tanzania": "TZ",
    "Macedonia": "MK",
    "Syria": "SY",
    "Laos": "LA",
    "Palestinian Territories": "PS",
    "Congo (Kinshasa)": "CD",
    "Congo (Brazzaville)": "CG",
    "Ivory Coast": "CI",
    "Swaziland": "SZ",
    "Turkey": "TR",    # en versiones nuevas de pycountry el país se llama "Türkiye"
}

def nombre_a_iso2(nombre):
    # Devuelve el código de 2 letras de un país, o None si no lo encuentra
    nombre = nombre.strip()
    if nombre in EXCEPCIONES:
        return EXCEPCIONES[nombre]
    try:
        return pycountry.countries.lookup(nombre).alpha_2  #.alpha_2 obtiene el código ISO de dos letras
    except LookupError:
        return None

happiness = q("""
    SELECT country AS country_name, year, "Life Ladder" AS life_ladder
    FROM happiness
    WHERE year BETWEEN 2005 AND 2014
""")
#apply() aplicamos la función nombre_a_iso2 a cada valor de la columna country_name
#La función devuelve el código de dos letras y lo guardamos en una nueva columna llamada country
happiness["country"] = happiness["country_name"].apply(nombre_a_iso2)


# Países a los que no les encontramos código
# happiness[happiness["country"] filtra las filas que no tienen código de país.
sin_codigo = sorted(happiness[happiness["country"].isna()]["country_name"].unique())
print("Países sin código:", sin_codigo)

Países sin código: ['Kosovo', 'North Cyprus', 'Somaliland region']


Revisamos la lista de países sin código. Si aparece alguno que también está en `hofstede` o tiene
usuarios, hay que agregarlo a `EXCEPCIONES`.

In [ ]:
happiness = happiness.dropna(subset=["country"]) #eliminamos las filas que quedaron sin código de país

# Registramos el DataFrame en DuckDB para poder guardarlo con la misma función de siempre
#con.register() hacemos que DuckDB pueda consultar ese DataFrame mediante SQL, usando el nombre happiness_codigo.
con.register("happiness_codigo", happiness)
guardar("SELECT country, country_name, year, life_ladder FROM happiness_codigo", "happiness_clean")

happiness_clean: 1,121 filas guardadas en c:/Users/mgrau/musica-por-pais/data/clean/happiness_clean.parquet


In [ ]:
#Verificamos que todos los países de `hofstede` tengan datos de happiness
q(f"""
    SELECT country, country_name
    FROM '{ruta_clean}hofstede_clean.parquet'
    WHERE country NOT IN (SELECT country FROM '{ruta_clean}happiness_clean.parquet')
""")

,country,country_name


Si la tabla esta vacia significa que todos los países de Hofstede tienen al menos un registro en Happiness durante el período analizado.

## 7. Escuchas

**Período válido.** como los datos van de 2005 a 2014 y el timestamp
está expresado en segundos desde 1970, calculamos en segundos el inicio
de 2005 y el de 2015 para poder filtrar las escuchas.

**Días anómalos.** En la exploración tomamos 720 escuchas por día como
límite para detectar actividad fuera de lo normal. Para agrupar las
escuchas por día usamos timestamp // 86400, ya que un día tiene
86.400 segundos.

**Escuchas limpias.** eliminamos las escuchas que están fuera del período y las
de los días en los que un usuario superó las 720 escuchas. También
sacamos la columna album_id porque no la vamos a usar.

In [ ]:
# Período válido: del 1/1/2005 al 31/12/2014, en UTC
#.timestamp(): convierte la fecha a segundos desde 1970.
inicio = int(pd.Timestamp("2005-01-01", tz="UTC").timestamp())
fin = int(pd.Timestamp("2015-01-01", tz="UTC").timestamp())    # primer segundo de 2015, queda afuera
print("inicio:", inicio, " fin:", fin)

inicio: 1104537600  fin: 1420070400


In [20]:
# Días en los que un usuario tuvo más de 720 escuchas
guardar("""
    SELECT user_id, "timestamp" // 86400 AS dia, COUNT(*) AS escuchas
    FROM events
    GROUP BY user_id, dia
    HAVING COUNT(*) > 720
""", "dias_anomalos")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dias_anomalos: 664 filas guardadas en c:/Users/mgrau/musica-por-pais/data/clean/dias_anomalos.parquet


In [ ]:
#Contamos los días anómalos, las escuchas de esos días y los usuarios involucrados.
q(f"""
    SELECT COUNT(*) AS dias_anomalos,
           SUM(escuchas) AS escuchas_en_esos_dias,
           COUNT(DISTINCT user_id) AS usuarios
    FROM '{ruta_clean}dias_anomalos.parquet'
""")

,dias_anomalos,escuchas_en_esos_dias,usuarios
0,664,727379.0,140


In [ ]:

#El LEFT JOIN relaciona la tabla events con el archivo dias_anomalos.parquet.
#para que haya coincidencia se deben cumplir dos condiciones: que sea el mismo usuario y que la escucha
#corresponda al mismo dia
guardar(f"""
    SELECT e.user_id, e.artist_id, e.track_id, e."timestamp"
    FROM events e
    LEFT JOIN read_parquet('{ruta_clean}dias_anomalos.parquet') d
           ON e.user_id = d.user_id AND e."timestamp" // 86400 = d.dia
    WHERE d.user_id IS NULL
      AND e."timestamp" >= {inicio}
      AND e."timestamp" < {fin}
""", "events_clean")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

events_clean: 350,741,953 filas guardadas en c:/Users/mgrau/musica-por-pais/data/clean/events_clean.parquet


In [23]:
# La primera y la última escucha ya no deberían ser la de 1970
q(f"""
    SELECT to_timestamp(MIN("timestamp")) AS primera_escucha,
           to_timestamp(MAX("timestamp")) AS ultima_escucha
    FROM '{ruta_clean}events_clean.parquet'
""")

,primera_escucha,ultima_escucha
0,2005-02-14 00:01:56+00:00,2014-09-01 23:56:08+00:00


## 8. Verificación general

In [24]:
#execute(): ejecuta la consulta SQL.
#fetchone(): obtiene la primera fila del resultado como una tupla.
filas = []
for original, limpia in [("events", "events_clean"), ("users", "users_clean"),
                         ("acoustic_features", "acoustic_features_clean"),
                         ("hofstede", "hofstede_clean"), ("happiness", "happiness_clean")]:
    antes = con.execute(f"SELECT COUNT(*) FROM {original}").fetchone()[0]
    despues = con.execute(f"SELECT COUNT(*) FROM '{ruta_clean}{limpia}.parquet'").fetchone()[0]
    filas.append({"tabla": original, "filas antes": antes, "filas después": despues, "diferencia": antes - despues})

pd.DataFrame(filas)

,tabla,filas antes,filas después,diferencia
0,events,351469333,350741953,727380
1,users,55190,55190,0
2,acoustic_features,3471884,3471884,0
3,hofstede,47,47,0
4,happiness,1562,1121,441
